In [3]:
import os
import math
import pickle
import folium
import pickle
import numpy as np
import pandas as pd
import rustworkx as rx
import geopandas as gpd

from tqdm import tqdm

In [4]:
school_data = pd.read_csv("data/ofsted_bradford_data.csv")
urn_name_map = school_data.set_index("URN")["School name"].to_dict()

In [14]:
school_data

,URN,School name,Ofsted phase,Type of education,School open date,Admissions policy,Sixth form,Designated religious character,Religious ethos,Faith grouping,...,Early years provision (where applicable),Sixth form provision (where applicable),Name,Lat,Lon,Website,Attainment 8,Progress 8,In Bradford,Fair banding
0,107395,Titus Salt School,Secondary,Community School,NaN,Non-selective,Has a sixth form,Does not apply,Does not apply,Non-faith,...,9,2,Titus Salt School,53.842914,-1.794062,https://www.titussaltschool.co.uk/,39.1,-0.58,True,False
1,107439,Bingley Grammar School,Secondary,Voluntary Aided School,01/01/1900,Non-selective,Has a sixth form,NaN,Does not apply,Non-faith,...,9,9,Bingley Grammar School,53.855390,-1.846753,https://www.bingleygrammar.org/,48.9,-0.03,True,False
2,107802,Fairfield School,Special,Foundation Special School,NaN,Not applicable,Has a sixth form,Does not apply,Does not apply,Non-faith,...,9,9,Fairfield School,53.720996,-1.665745,https://www.fairfieldschool.uk/,NaN,NaN,False,False
3,108079,Pudsey Grammar School,Secondary,Foundation School,NaN,Non-selective,Has a sixth form,Does not apply,Does not apply,Non-faith,...,9,2,Pudsey Grammar School,53.800409,-1.659174,https://www.pudseygrammar.co.uk/,45.5,-0.12,False,False
4,108083,Benton Park School,Secondary,Community School,NaN,Non-selective,Has a sixth form,Does not apply,Does not apply,Non-faith,...,9,2,Benton Park School,53.856965,-1.682909,https://www.bentonpark.org.uk/,46.8,-0.15,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99,139152,Great Sankey High School,Secondary,Academy Converter,01/01/2013,Non-selective,Has a sixth form,Does not apply,Does not apply,Non-faith,...,9,2,Great Sankey High School,53.400290,-2.661213,NaN,NaN,NaN,False,False
100,139196,Beamont Collegiate Academy,Secondary,Academy Sponsor Led,01/03/2013,Non-selective,Does not have a sixth form,Does not apply,NaN,Non-faith,...,9,9,Beamont Collegiate Academy,53.407684,-2.587194,NaN,NaN,NaN,False,False
101,139474,One In A Million Free School,Secondary,Free School,02/09/2013,Non-selective,Does not have a sixth form,Does not apply,NaN,Non-faith,...,9,9,One In A Million Free School,53.805129,-1.760460,NaN,NaN,NaN,True,False
102,141307,Ryburn Valley High School,Secondary,Academy Converter,01/09/2014,Non-selective,Has a sixth form,NaN,Does not apply,Non-faith,...,9,9,Ryburn Valley High School,53.703992,-1.933347,NaN,NaN,NaN,False,False


In [5]:
# Load geospatial data
oa21lsoa = pd.read_excel("data/OA11_LAD21_LSOA11_MSOA11_LEP21_EN_v3.xlsx")
oa_gdf = gpd.read_file("data/Output_Areas_Dec_2011_Boundaries_EW_BGC_2022_-9162807391755440229.geojson" )

# Select only OAs in and around Bradford
selected_oas = oa21lsoa.loc[oa21lsoa["LAD21NM"].isin(["Bradford", "Leeds", "Calderdale", "Kirklees", "Craven", "Harrogate"]), "OA11CD"].unique()
bradford_gdf = oa_gdf.loc[oa_gdf["OA11CD"].isin(selected_oas)]

# Get the centroid latitude and longitude coords for each OA
oa_unique = bradford_gdf.drop_duplicates(subset=["OA11CD"])
centroids_4326 = oa_unique.to_crs(epsg=27700).geometry.centroid.to_crs(epsg=4326)
oa_centroid_dict = dict(zip(oa_unique["OA11CD"], centroids_4326))

In [6]:
with open("transit_graph_.pkl", "rb") as f:
    G = pickle.load(f)

with open("node_to_idx_.pkl", "rb") as f:
    node_to_idx = pickle.load(f)

In [7]:
LON_SCALE_FACTOR = math.cos(math.radians(53.8))

In [11]:
OUTPUT_CSV = "school_oa_bus_durations_testing.csv"

file_exists = os.path.exists(OUTPUT_CSV)

COMPLETED_SCHOOLS = set(pd.read_csv(OUTPUT_CSV)["School_Name"]) if file_exists else {}

In [ ]:
for index, school in school_data.iterrows():
    school_name = school["Name"]

    if school_name in COMPLETED_SCHOOLS:
        print(f"[{school_name}] already completed, skipping.")
        continue
    school_urn = school["URN"]

    print(f"\nCalculating routes for: {school_name}...")
    school_records = []

    dest_idx = node_to_idx[school_urn]
    dest_node = G[dest_idx]

    def gcs_manhattan_node(node):
        return abs(dest_node["lat"] - node["lat"]) + (abs(dest_node["lon"] - node["lon"]) * LON_SCALE_FACTOR)

    for oa in tqdm(selected_oas[:100]):
        path_indices = rx.astar_shortest_path(
            G,
            node=node_to_idx[oa],
            goal_fn=lambda node: (node["name"] == school_urn),
            edge_cost_fn=lambda edge: edge["weight"],
            estimate_cost_fn=gcs_manhattan_node
        )

        optimal_path = [G.get_node_data(i)["name"] for i in path_indices]

        optimal_duration = 0.0
        for i in range(len(path_indices) - 1):
            edge_data = G.get_edge_data(path_indices[i], path_indices[i+1])
            optimal_duration += edge_data["weight"]

        oa_cent = oa_centroid_dict[oa]
        school_records.append({
                    "School_Name": school_name,
                    "School_URN": school_urn,
                    "School_Lat": school["Lat"],
                    "School_Lon": school["Lon"],
                    "OA_Code": oa,
                    "OA_Centroid_Y": oa_cent.y,
                    "OA_Centroid_X": oa_cent.x,
                    "Duration_Seconds": optimal_duration,
                    "Path": optimal_path
                })

    school_df = pd.DataFrame(school_records)
    school_df.to_csv(OUTPUT_CSV, mode='a', index=False, header=not file_exists)
    file_exists = True 
    
    print(f"Successfully saved data for {school_name} to {OUTPUT_CSV}")

school_df

[Titus Salt School] already completed, skipping.
[Bingley Grammar School] already completed, skipping.
[Fairfield School] already completed, skipping.
[Pudsey Grammar School] already completed, skipping.
[Benton Park School] already completed, skipping.
[Guiseley School] already completed, skipping.
[Brooklands School] already completed, skipping.
[Dixons City Academy] already completed, skipping.
[West Specialist Inclusive Learning Centre (Mainsite)] already completed, skipping.
[Bradford Academy] already completed, skipping.
[Craven Pupil Referral Service] already completed, skipping.
[Dixons Allerton Academy] already completed, skipping.
[Leeds West Academy] already completed, skipping.
[Trinity Academy Halifax] already completed, skipping.
[Carlton Keighley] already completed, skipping.
[Brighouse High School] already completed, skipping.
[South Craven School] already completed, skipping.
[The North Halifax Grammar School] already completed, skipping.
[Ilkley Grammar School] alread

NameError: name 'school_df' is not defined